# Zero-Shot Prompting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/01-foundational/01_zero_shot_prompting.ipynb)

**Category:** 01 - Foundational Prompting  **Technique #:** 01  **Difficulty:** Beginner

## Description

Zero-shot prompting is the most fundamental prompting technique where you ask the model to perform a task **without providing any examples**. The model relies entirely on its pre-trained knowledge to understand and execute the request.

### When to Use:
- Simple, well-defined tasks
- When the task is commonly understood
- Quick prototyping and experimentation
- Tasks where examples might bias the output
- When you want to test the model's base capabilities

### When NOT to Use:
- Complex tasks requiring specific formatting
- When consistent output structure is critical
- Domain-specific tasks with unique conventions
- When you need the model to follow a specific pattern

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                     ZERO-SHOT FLOW                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   [User] ──> Task Description ──> [LLM]                     │
│                                      │                      │
│                                      ▼                      │
│                              Pre-trained Knowledge          │
│                                      │                      │
│                                      ▼                      │
│   [User] <── Response <───────── Output Generation          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Key Characteristics:
1. **No examples provided** - Just the instruction
2. **Relies on pre-training** - Model uses learned patterns
3. **Simple and fast** - No preparation needed
4. **Flexible** - Model decides the approach

### The Prompt Structure:
```
[Instruction/Question] + [Task Description] + [Input Data]
```

## Setup

Install required packages and set up API access.

In [ ]:
# Install required packages
!pip install openai -q

# For Google Colab: Upload your API key securely
from getpass import getpass
import os

# Get API key securely (won't be displayed)
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

print("✓ Setup complete!")

## Basic Example

Let's start with a simple zero-shot classification task.

In [ ]:
def zero_shot_classify(text):
    """
    Zero-shot sentiment classification without any examples.
    """
    prompt = f"""
Classify the sentiment of the following text as positive, negative, or neutral.

Text: {text}
Sentiment:
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,  # Lower temperature for more consistent results
        max_tokens=50
    )
    
    return response.choices[0].message.content.strip()

# Test with different texts
test_texts = [
    "I absolutely love this new phone! The camera is amazing.",
    "The weather today is quite cloudy.",
    "This service was terrible and the staff was rude."
]

print("Zero-Shot Sentiment Classification Results:")
print("=" * 50)
for text in test_texts:
    sentiment = zero_shot_classify(text)
    print(f"\nText: {text}")
    print(f"Sentiment: {sentiment}")

## Real-World Example

A practical application: Email categorization for a customer support system.

In [ ]:
def categorize_support_email(email_body):
    """
    Categorize customer support emails into predefined categories.
    This is used in an automated ticket routing system.
    """
    prompt = f"""
Categorize this customer support email into one of these categories:
- Billing Issue
- Technical Support
- Account Access
- Feature Request
- Complaint
- General Inquiry

Email:
{email_body}

Category (respond with only the category name):
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=30
    )
    
    return response.choices[0].message.content.strip()

# Sample support emails
emails = [
    """
    Hi, I was charged twice for my subscription this month. 
    Can you please refund the duplicate charge? 
    My account ID is ACC-12345.
    """,
    """
    The app keeps crashing whenever I try to upload photos. 
    I've tried reinstalling but the issue persists. 
    Using iPhone 14 on iOS 17.
    """,
    """
    I forgot my password and can't access the verification email 
    since I no longer have access to my old email address.
    """
]

print("Support Email Categorization:")
print("=" * 60)
for i, email in enumerate(emails, 1):
    category = categorize_support_email(email)
    print(f"\nEmail {i}:")
    print(f"Category: {category}")
    print(f"Preview: {email[:80]}...")

## Failure Case

Zero-shot prompting can fail when the task requires specific domain knowledge or consistent formatting.

In [ ]:
# Example where zero-shot fails: Complex structured output

def extract_entities_zero_shot(text):
    """
    Attempting to extract entities without examples.
    The output format may be inconsistent.
    """
    prompt = f"""
Extract all named entities from this text and categorize them.

Text: {text}

Entities:
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=200
    )
    
    return response.choices[0].message.content.strip()

# Test with multiple similar texts
test_texts = [
    "Apple CEO Tim Cook announced new products in Cupertino.",
    "Microsoft CEO Satya Nadella spoke at the Seattle conference.",
    "Google's Sundar Pichai presented in Mountain View."
]

print("Zero-Shot Entity Extraction (Inconsistent Results):")
print("=" * 60)
for text in test_texts:
    entities = extract_entities_zero_shot(text)
    print(f"\nInput: {text}")
    print(f"Output: {entities}")
    print("-" * 40)

print("\n⚠️ Notice how the output format varies between responses!")
print("This is a limitation of zero-shot prompting for structured tasks.")

## Benchmark

### Zero-Shot Performance Comparison

| Task Type | Model | Zero-Shot Accuracy | Notes |
|-----------|-------|-------------------|-------|
| Sentiment Analysis | GPT-3.5 | 85-90% | Good for simple texts |
| Text Classification | GPT-3.5 | 80-85% | Depends on categories |
| Named Entity Recognition | GPT-3.5 | 70-75% | Format inconsistency |
| Translation | GPT-4 | 90-95% | Excellent for major languages |
| Summarization | GPT-4 | 85-90% | Quality varies by length |
| Question Answering | GPT-4 | 75-85% | Depends on domain |

### Key Insights:
- **Strengths:** Quick to implement, no data preparation needed
- **Weaknesses:** Inconsistent formatting, limited control
- **Best For:** Simple, well-understood tasks
- **Improvement:** Add examples (few-shot) for +10-20% accuracy

## Interactive Playground

Experiment with zero-shot prompting below!

In [ ]:
# Interactive Zero-Shot Playground

def zero_shot_playground(task_description, input_text, model="gpt-3.5-turbo"):
    """
    Generic zero-shot prompting function.
    
    Args:
        task_description: What you want the model to do
        input_text: The input to process
        model: Which model to use
    """
    prompt = f"""{task_description}

{input_text}
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=300
    )
    
    return response.choices[0].message.content.strip()

# ═══════════════════════════════════════════════════════
# MODIFY THESE VARIABLES TO EXPERIMENT
# ═══════════════════════════════════════════════════════

my_task = """Translate the following text to French.
Maintain the tone and style of the original."""

my_input = "The quick brown fox jumps over the lazy dog."

# Run the experiment
result = zero_shot_playground(my_task, my_input)
print("Result:")
print(result)

# Try other tasks:
# - "Summarize this article in 3 bullet points"
# - "Extract keywords from this text"
# - "Rewrite this in a professional tone"
# - "Identify the main topic of this paragraph"

## Tips & Tricks

### Model-Specific Advice

**GPT-3.5 Turbo:**
- Works well for simple classification tasks
- Use temperature=0.2-0.3 for consistency
- May need clearer instructions than GPT-4

**GPT-4:**
- Excellent zero-shot performance
- Better at following complex instructions
- More consistent output formatting

**Claude (Anthropic):**
- Strong at reasoning tasks zero-shot
- Good at explaining its thought process

**Gemini (Google):**
- Competitive zero-shot capabilities
- Good multimodal zero-shot performance

### Best Practices

1. **Be Specific** - Clear instructions beat vague ones
2. **Use Action Verbs** - "Classify", "Extract", "Summarize"
3. **Define Output Format** - Even without examples, specify format
4. **Set Temperature Low** - For consistent, deterministic results
5. **Iterate** - Test and refine your prompt

### Common Pitfalls

- ❌ Vague instructions: "Analyze this"
- ✅ Specific instructions: "Analyze this text and identify the top 3 themes"

- ❌ No output specification
- ✅ Clear format: "Respond with only the category name"

- ❌ Assuming domain knowledge
- ✅ Providing context: "In the context of medical terminology..."

## References

### Academic Papers

1. **Language Models are Few-Shot Learners** (Brown et al., 2020)
   - [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
   - Introduced the concept of in-context learning including zero-shot

2. **How Can We Know What Language Models Know?** (Jiang et al., 2020)
   - [arXiv:1911.12543](https://arxiv.org/abs/1911.12543)
   - Analysis of zero-shot knowledge retrieval

### Documentation

- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Claude Documentation](https://docs.anthropic.com/claude/docs/intro-to-prompting)

### Related Techniques

- **Few-Shot Prompting** - Add examples for better consistency
- **Chain-of-Thought** - For complex reasoning tasks
- **System Prompting** - Set behavior context